#### Using data olist from kaggle dataset "olistbr/brazilian-ecommerce"

### Work directory: projects/DE/olist-analysis
### Environment

Activate project virtual environment in zsh/bash

``` source .venv/bin/activate ```

Check interpreter

``` which python ```

In [1]:
import sys

print(sys.executable)

/Users/zhw/projects/DE/olist-analysis/.venv/bin/python


#### import python package, load data path

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"


In [6]:
# customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))

csv_files = sorted(DATA_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_PATH}")

# tables = {}
# for csv_file in csv_files:
#     tables[csv_file.stem] = pd.read_csv(csv_file)

tables = {
    csv_file.stem: pd.read_csv(csv_file)
    for csv_file in csv_files
}

tables.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

#### now load all data table

In [ ]:
table_summary = []

for table_name, df in tables.items():
    table_summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    })

# table_summary = pd.DataFrame(table_summary)
# table_summary = table_summary.sort_values("rows", ascending=False)
# table_summary = table_summary.reset_index(drop=True)

table_summary = (
    pd.DataFrame(table_summary)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

table_summary

,table,rows,columns,duplicate_rows,missing_values,memory_mb
0,olist_geolocation_dataset,1000163,5,261831,0,130.264880
1,olist_order_items_dataset,112650,7,0,0,35.989649
2,olist_order_payments_dataset,103886,5,0,0,16.229413
3,olist_customers_dataset,99441,5,0,0,26.586405
4,olist_orders_dataset,99441,8,0,4908,52.937277
5,olist_order_reviews_dataset,99224,7,0,145903,39.124777
6,olist_products_dataset,32951,9,0,2448,6.296564
7,olist_sellers_dataset,3095,4,0,0,0.588103
8,product_category_name_translation,71,2,0,0,0.008999


In [ ]:
table_summary = pd.DataFrame([
    {
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    }
    for table_name, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)

table_summary

In [8]:
customers = tables["olist_customers_dataset"]
geolocation = tables["olist_geolocation_dataset"]
orders = tables["olist_orders_dataset"]
order_items = tables["olist_order_items_dataset"]
order_payments = tables["olist_order_payments_dataset"]
order_reviews = tables["olist_order_reviews_dataset"]
products = tables["olist_products_dataset"]
sellers = tables["olist_sellers_dataset"]
product_category_name_translation = tables["product_category_name_translation"]

#### next check the data structure

In [25]:
for table_name, df in tables.items():
    print("="*50)
    print(f"Table: {table_name}") # f-string allow variable interpolation
    print(df.shape)
    # print(df.head())
    df.info()
    print(f"null values: \n{df.isna().sum()}")

Table: olist_customers_dataset
(99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
null values: 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Table: olist_geolocation_dataset
(1000163, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  


In [ ]:
## Column Profiling

column_profile = pd.DataFrame([
    {
        "table": table_name,
        "column": col,
        "dtype": str(df[col].dtype),
        "rows": df.shape[0],
        "n_unique": df[col].nunique(),
        "null_count": df[col].isna().sum(),
        "null_pct": df[col].isna().mean() * 100,
        "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else None
    }
    for table_name, df in tables.items()
    for col in df.columns
])


In [113]:
column_profile[
    column_profile["table"] == "olist_order_reviews_dataset"
]

,table,column,dtype,rows,n_unique,null_count,null_pct,sample_value
22,olist_order_reviews_dataset,review_id,str,99224,98410,0,0.000000,7bc2406110b926393aa56f80a40eba40
23,olist_order_reviews_dataset,order_id,str,99224,98673,0,0.000000,73fc7af87114b39712e6da79b0a377eb
24,olist_order_reviews_dataset,review_score,int64,99224,5,0,0.000000,4
25,olist_order_reviews_dataset,review_comment_title,str,99224,4527,87656,88.341530,recomendo
26,olist_order_reviews_dataset,review_comment_message,str,99224,36159,58247,58.702532,Recebi bem antes do prazo estipulado.
27,olist_order_reviews_dataset,review_creation_date,str,99224,636,0,0.000000,2018-01-18 00:00:00
28,olist_order_reviews_dataset,review_answer_timestamp,str,99224,98248,0,0.000000,2018-01-18 21:46:59


In [96]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [77]:
customers["customer_zip_code_prefix"].value_counts()

customer_zip_code_prefix
22790    142
24220    124
22793    121
24230    117
22775    110
        ... 
87145      1
98860      1
5538       1
74980      1
99043      1
Name: count, Length: 14994, dtype: int64

## Table Grain

Grain defines the real-world meaning of one row.

| Table | Grain / One row represents |
|---|---|
| `olist_customers_dataset` | One customer record |
| `olist_geolocation_dataset` | One geolocation record for a ZIP code prefix |
| `olist_orders_dataset` | One order |
| `olist_order_items_dataset` | One item record within an order |
| `olist_order_payments_dataset` | One payment record for an order |
| `olist_order_reviews_dataset` | One review record for an order |
| `olist_products_dataset` | One product |
| `olist_sellers_dataset` | One seller |
| `product_category_name_translation` | One product-category name translation mapping |

## Candidate Key Assumptions

Candidate keys are fields or combinations of fields that are expected to uniquely identify one row according to the table grain.

Superkey = any set of columns that uniquely identifies a row.

Candidate key = a minimal superkey that uniquely identifies a row.


| Table | Candidate Key Assumption | Reason |
|---|---|---|
| `olist_customers_dataset` | `customer_id` | Each row represents one customer record, and `customer_id` is expected to identify that record uniquely. |
| `olist_geolocation_dataset` | None identified | ZIP code prefixes are not unique, and multiple geolocation records may exist for the same prefix. |
| `olist_orders_dataset` | `order_id` | Each row represents one order. |
| `olist_order_items_dataset` | (`order_id`, `order_item_id`) | An order can contain multiple item records, so the item sequence is needed within each order. |
| `olist_order_payments_dataset` | (`order_id`, `payment_sequential`) | An order can have multiple payment records. |
| `olist_order_reviews_dataset` | `review_id` | Each row represents a review record, so `review_id` is expected to identify the review uniquely. |
| `olist_products_dataset` | `product_id` | Each row represents one product. |
| `olist_sellers_dataset` | `seller_id` | Each row represents one seller. |
| `product_category_name_translation` | `product_category_name` | Each source category is expected to have one English translation mapping. |

In [ ]:
## Candidate Key Validation

table_candidate_keys = {
    "olist_customers_dataset": ["customer_id"],
    "olist_geolocation_dataset": None,
    "olist_orders_dataset": ["order_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
}

records = []

for table_name, df in tables.items():
    keys = table_candidate_keys[table_name]
    row_count = len(df)

    if keys is None:
        records.append({
            "table": table_name,
            "column_or_composite": "None identified",
            "key_type": "None",
            "row_count": row_count,
            "unique_count": None,
            "duplicate_count": None,
            "null_count": None,
            "is_unique": None,
        })
        continue

    # Validate each component independently
    for col in keys:
        unique_count = df[col].nunique(dropna=True)
        duplicate_count = df.duplicated(
            subset=[col],
            keep="first"
        ).sum()
        null_count = df[col].isna().sum()

        records.append({
            "table": table_name,
            "column_or_composite": col,
            "key_type": "Single Column",
            "row_count": row_count,
            "unique_count": unique_count,
            "duplicate_count": duplicate_count,
            "null_count": null_count,
            "is_unique": (
                unique_count == row_count
                and null_count == 0
            ),
        })

    # Validate the composite key
    if len(keys) > 1:
        unique_count = (
            df[keys]
            .drop_duplicates()
            .shape[0]
        )

        duplicate_count = df.duplicated(
            subset=keys,
            keep="first"
        ).sum()

        null_count = (
            df[keys]
            .isna()
            .any(axis=1)
            .sum()
        )

        records.append({
            "table": table_name,
            "column_or_composite": " + ".join(keys),
            "key_type": "Composite Key",
            "row_count": row_count,
            "unique_count": unique_count,
            "duplicate_count": duplicate_count,
            "null_count": null_count,
            "is_unique": (
                unique_count == row_count
                and null_count == 0
            ),
        })

candidate_key_validation = pd.DataFrame(records)

candidate_key_validation

,table,column_or_composite,key_type,row_count,unique_count,duplicate_count,null_count,is_unique
0,olist_customers_dataset,customer_id,Single Column,99441,99441.0,0.0,0.0,True
1,olist_geolocation_dataset,None identified,None,1000163,NaN,NaN,NaN,None
2,olist_order_items_dataset,order_id,Single Column,112650,98666.0,13984.0,0.0,False
3,olist_order_items_dataset,order_item_id,Single Column,112650,21.0,112629.0,0.0,False
4,olist_order_items_dataset,order_id + order_item_id,Composite Key,112650,112650.0,0.0,0.0,True
5,olist_order_payments_dataset,order_id,Single Column,103886,99440.0,4446.0,0.0,False
6,olist_order_payments_dataset,payment_sequential,Single Column,103886,29.0,103857.0,0.0,False
7,olist_order_payments_dataset,order_id + payment_sequential,Composite Key,103886,103886.0,0.0,0.0,True
8,olist_order_reviews_dataset,review_id,Single Column,99224,98410.0,814.0,0.0,False
9,olist_orders_dataset,order_id,Single Column,99441,99441.0,0.0,0.0,True


In [ ]:
## investigate duplicated review_id in Table: olist_order_reviews_dataset
### Candidate key 是能够唯一标识当前 table grain 的最小字段集合

review_id_counts = (
    order_reviews["review_id"]
    .value_counts()
)

review_id_counts.head()


review_id
c444278834184f72b1484dfe47de7f97    3
308316408775d1600dad81bd3184556d    3
2d6ac45f859465b5c185274a1c929637    3
3415c9f764e478409e8e0660ae816dd2    3
4219a80ab469e3fc9901437b73da3f75    3
Name: count, dtype: int64

In [121]:
duplicated_review_ids = review_id_counts[
    review_id_counts > 1
].index

dup_reviews = order_reviews[
    order_reviews["review_id"].isin(duplicated_review_ids)
].sort_values("review_id")

dup_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [125]:
order_reviews.groupby("review_id")["order_id"].nunique().value_counts()

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

In [126]:
order_reviews.groupby("order_id")["review_id"].nunique().value_counts()

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

In [127]:
order_reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

np.int64(0)

In [128]:
order_reviews[
    ["review_id", "order_id"]
].isna().sum()

review_id    0
order_id     0
dtype: int64